In [0]:
print("Spark Session Started")
print(spark)

Spark Session Started


In [0]:
from pyspark.sql.types import StructType, StructField, IntegerType, StringType
from pyspark.sql.functions import col

In [0]:
# Define schema for ratings dataset
ratings_schema = StructType([
    StructField("user_id", IntegerType(), True),
    StructField("movie_id", IntegerType(), True),
    StructField("rating", IntegerType(), True),
    StructField("timestamp", StringType(), True)
])

In [0]:
# Create sample ratings data
ratings_data = [
    (1, 101, 5, "2024-01-01"),
    (2, 101, 4, "2024-01-02"),
    (3, 102, 3, "2024-01-03"),
    (4, 103, 2, "2024-01-04"),
    (5, 104, 5, "2024-01-05"),
    (6, 105, 4, "2024-01-06"),
    (7, 102, 6, "2024-01-07"),      # Invalid rating
    (8, 103, None, "2024-01-08"),   # Null rating
    (9, 101, 5, "2024-01-09"),
    (1, 101, 5, "2024-01-01")       # Duplicate row
]

In [0]:
# Create ratings DataFrame
ratings_df = spark.createDataFrame(ratings_data, ratings_schema)

# Display ratings data
display(ratings_df)

user_id,movie_id,rating,timestamp
1,101,5,2024-01-01
2,101,4,2024-01-02
3,102,3,2024-01-03
4,103,2,2024-01-04
5,104,5,2024-01-05
6,105,4,2024-01-06
7,102,6,2024-01-07
8,103,null,2024-01-08
9,101,5,2024-01-09
1,101,5,2024-01-01


In [0]:
# Define schema for movies dataset
movies_schema = StructType([
    StructField("movie_id", IntegerType(), True),
    StructField("movie_name", StringType(), True),
    StructField("genre", StringType(), True),
    StructField("release_year", IntegerType(), True)
])

# Create sample movie data
movies_data = [
    (101, "Avatar", "Action", 2009),
    (102, "Titanic", "Romance", 1997),
    (103, "Joker", "Drama", 2019),
    (104, "Avengers", "Action", 2012),
    (105, "Inception", "Sci-Fi", 2010)
]

# Create movies DataFrame
movies_df = spark.createDataFrame(movies_data, movies_schema)

# Display movies data
display(movies_df)

movie_id,movie_name,genre,release_year
101,Avatar,Action,2009
102,Titanic,Romance,1997
103,Joker,Drama,2019
104,Avengers,Action,2012
105,Inception,Sci-Fi,2010


In [0]:
print("Ratings Schema:")
ratings_df.printSchema()

print("Movies Schema:")
movies_df.printSchema()

Ratings Schema:
root
 |-- user_id: integer (nullable = true)
 |-- movie_id: integer (nullable = true)
 |-- rating: integer (nullable = true)
 |-- timestamp: string (nullable = true)

Movies Schema:
root
 |-- movie_id: integer (nullable = true)
 |-- movie_name: string (nullable = true)
 |-- genre: string (nullable = true)
 |-- release_year: integer (nullable = true)



In [0]:
print("Total records in ratings dataset:", ratings_df.count())
print("Total records in movies dataset:", movies_df.count())

Total records in ratings dataset: 10
Total records in movies dataset: 5


In [0]:

# Find rows with null values in ratings dataset
null_ratings_df = ratings_df.filter(
    col("user_id").isNull() |
    col("movie_id").isNull() |
    col("rating").isNull() |
    col("timestamp").isNull()
)

display(null_ratings_df)

user_id,movie_id,rating,timestamp
8,103,null,2024-01-08


In [0]:
invalid_rating_df = ratings_df.filter(
    (col("rating") < 1) | (col("rating") > 5)
)

display(invalid_rating_df)

user_id,movie_id,rating,timestamp
7,102,6,2024-01-07


In [0]:
clean_ratings_df = ratings_df.dropna()

clean_ratings_df = clean_ratings_df.filter(
    (col("rating") >= 1) & (col("rating") <= 5)
)

clean_ratings_df = clean_ratings_df.dropDuplicates([
    "user_id",
    "movie_id",
    "rating",
    "timestamp"
])

display(clean_ratings_df)

user_id,movie_id,rating,timestamp
1,101,5,2024-01-01
2,101,4,2024-01-02
3,102,3,2024-01-03
4,103,2,2024-01-04
5,104,5,2024-01-05
6,105,4,2024-01-06
9,101,5,2024-01-09


In [0]:
print("Total records before cleaning:", ratings_df.count())
print("Total records after cleaning:", clean_ratings_df.count())

Total records before cleaning: 10
Total records after cleaning: 7


In [0]:
print("Total records before cleaning:", ratings_df.count())
print("Total records after cleaning:", clean_ratings_df.count())

Total records before cleaning: 10
Total records after cleaning: 7


In [0]:
movie_ratings_df = clean_ratings_df.join(
    movies_df,
    on="movie_id",
    how="inner"
)

display(movie_ratings_df)

movie_id,user_id,rating,timestamp,movie_name,genre,release_year
101,1,5,2024-01-01,Avatar,Action,2009
101,2,4,2024-01-02,Avatar,Action,2009
102,3,3,2024-01-03,Titanic,Romance,1997
103,4,2,2024-01-04,Joker,Drama,2019
104,5,5,2024-01-05,Avengers,Action,2012
105,6,4,2024-01-06,Inception,Sci-Fi,2010
101,9,5,2024-01-09,Avatar,Action,2009


In [0]:
from pyspark.sql.functions import avg, count

average_rating_df = movie_ratings_df.groupBy(
    "movie_id",
    "movie_name",
    "genre"
).agg(
    avg("rating").alias("average_rating"),
    count("rating").alias("total_ratings")
)

display(average_rating_df)

movie_id,movie_name,genre,average_rating,total_ratings
101,Avatar,Action,4.666666666666667,3
102,Titanic,Romance,3.0,1
103,Joker,Drama,2.0,1
104,Avengers,Action,5.0,1
105,Inception,Sci-Fi,4.0,1


In [0]:
from pyspark.sql.functions import desc

popular_movies_df = average_rating_df.orderBy(
    desc("total_ratings")
)

display(popular_movies_df)

movie_id,movie_name,genre,average_rating,total_ratings
101,Avatar,Action,4.666666666666667,3
102,Titanic,Romance,3.0,1
103,Joker,Drama,2.0,1
104,Avengers,Action,5.0,1
105,Inception,Sci-Fi,4.0,1


In [0]:
genre_analysis_df = movie_ratings_df.groupBy(
    "genre"
).agg(
    avg("rating").alias("average_genre_rating"),
    count("rating").alias("total_ratings")
).orderBy(
    desc("total_ratings")
)

display(genre_analysis_df)

genre,average_genre_rating,total_ratings
Action,4.75,4
Romance,3.0,1
Drama,2.0,1
Sci-Fi,4.0,1


In [0]:
from pyspark.sql.functions import when

movie_quality_df = average_rating_df.withColumn(
    "movie_quality",
    when(col("average_rating") >= 4, "High Rated")
    .when(col("average_rating") >= 3, "Average Rated")
    .otherwise("Low Rated")
)

display(movie_quality_df)

movie_id,movie_name,genre,average_rating,total_ratings,movie_quality
101,Avatar,Action,4.666666666666667,3,High Rated
102,Titanic,Romance,3.0,1,Average Rated
103,Joker,Drama,2.0,1,Low Rated
104,Avengers,Action,5.0,1,High Rated
105,Inception,Sci-Fi,4.0,1,High Rated


In [0]:
movie_ratings_df.createOrReplaceTempView("movie_ratings")
average_rating_df.createOrReplaceTempView("average_movie_ratings")
popular_movies_df.createOrReplaceTempView("popular_movies")
genre_analysis_df.createOrReplaceTempView("genre_analysis")
movie_quality_df.createOrReplaceTempView("movie_quality_report")

print("Temporary views created successfully")

Temporary views created successfully


In [0]:
%sql
SELECT * FROM movie_quality_report;

movie_id,movie_name,genre,average_rating,total_ratings,movie_quality
101,Avatar,Action,4.666666666666667,3,High Rated
102,Titanic,Romance,3.0,1,Average Rated
103,Joker,Drama,2.0,1,Low Rated
104,Avengers,Action,5.0,1,High Rated
105,Inception,Sci-Fi,4.0,1,High Rated


In [0]:
%sql
SELECT genre, average_genre_rating, total_ratings
FROM genre_analysis
ORDER BY total_ratings DESC;

genre,average_genre_rating,total_ratings
Action,4.75,4
Romance,3.0,1
Drama,2.0,1
Sci-Fi,4.0,1


In [0]:
%sql
SELECT movie_name, total_ratings
FROM popular_movies
ORDER BY total_ratings DESC;

movie_name,total_ratings
Avatar,3
Titanic,1
Joker,1
Avengers,1
Inception,1


In [0]:
average_rating_df.write.mode("overwrite").saveAsTable("average_movie_ratings")
popular_movies_df.write.mode("overwrite").saveAsTable("popular_movies")
genre_analysis_df.write.mode("overwrite").saveAsTable("genre_analysis")
movie_quality_df.write.mode("overwrite").saveAsTable("movie_quality_report")

print("Output tables saved successfully")

Output tables saved successfully
